# XGBoost Parameter Tuning

Grid-search over XGBoost hyperparameters using a 48-hour holdout backtest on the EIA load data.

Because XGBoost is not a native time-series model, each series is turned into a supervised
learning problem with **lag**, **calendar**, and **rolling** features. Multi-step forecasts are
produced **recursively** (each prediction is fed back in as the next step's lag), mirroring how the
SARIMA notebook forecasts the full holdout window.

**How to use:**
1. Run **Cell 1** to install dependencies.
2. Run **Cell 2** to upload your `load_hourly.parquet` file.
3. Edit the `CONFIGURATION` cell to choose regions, features, and the hyperparameter search space.
4. Run all remaining cells. Results are sorted by MAPE ascending (lower is better).

In [ ]:
# Cell 1 - Install dependencies
!pip install -q xgboost scikit-learn pandas pyarrow tqdm matplotlib

In [ ]:
# Cell 2 - Upload the parquet file
from google.colab import files
import io

print("Upload your load_hourly.parquet file:")
uploaded = files.upload()

import pandas as pd

filename = list(uploaded.keys())[0]
load_df = pd.read_parquet(io.BytesIO(uploaded[filename]))
load_df["period"] = pd.to_datetime(load_df["period"], utc=True)
load_df["value"] = pd.to_numeric(load_df["value"], errors="coerce")

print(f"\nLoaded {len(load_df):,} rows")
print(f"Date range : {load_df['period'].min()} -> {load_df['period'].max()}")
print(f"Regions    : {sorted(load_df['region'].unique())}")
load_df.head()

In [ ]:
# ──────────────────────────────────────────────
# CONFIGURATION  <- edit this cell
# ──────────────────────────────────────────────

# Regions to include in the search. Set to None to use all regions.
REGIONS = None  # e.g. ["CISO", "ERCO"]

# How many recent hours to hold out for evaluation
HOLDOUT_HOURS = 48

# Max training history (None = use everything available)
MAX_TRAIN_HOURS = 1440  # ~60 days

# Minimum non-null observations required to fit the model
MIN_OBSERVATIONS = 96

# Feature engineering
LAGS = [1, 2, 3, 24, 25, 48, 168]        # hourly lags fed into the model
ROLL_WINDOWS = [24, 168]                  # rolling-mean windows (computed on shifted history)

# Hyperparameter search space (cartesian product is searched)
N_ESTIMATORS_VALUES   = [300, 600]
MAX_DEPTH_VALUES      = [4, 6]
LEARNING_RATE_VALUES  = [0.03, 0.1]
SUBSAMPLE_VALUES      = [0.9]
COLSAMPLE_VALUES      = [0.9]
MIN_CHILD_WEIGHT_VALUES = [1, 5]

# Fixed params applied to every model
FIXED_PARAMS = {
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": 42,
}
# ──────────────────────────────────────────────

In [ ]:
# Feature engineering + recursive XGBoost forecast helper
import warnings
import numpy as np
from dataclasses import dataclass, field
from xgboost import XGBRegressor

CALENDAR_COLS = [
    "hour", "dayofweek", "month", "is_weekend",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
]
LAG_COLS = [f"lag_{l}" for l in LAGS]
ROLL_COLS = [f"rollmean_{w}" for w in ROLL_WINDOWS]
FEATURE_COLS = CALENDAR_COLS + LAG_COLS + ROLL_COLS


def make_features(ts: pd.Series) -> pd.DataFrame:
    """Build a supervised feature frame from an hourly series indexed by timestamp."""
    df = pd.DataFrame({"y": ts.astype(float)})
    idx = df.index
    df["hour"] = idx.hour
    df["dayofweek"] = idx.dayofweek
    df["month"] = idx.month
    df["is_weekend"] = (idx.dayofweek >= 5).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
    df["dow_sin"] = np.sin(2 * np.pi * idx.dayofweek / 7)
    df["dow_cos"] = np.cos(2 * np.pi * idx.dayofweek / 7)
    for lag in LAGS:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    for w in ROLL_WINDOWS:
        df[f"rollmean_{w}"] = df["y"].shift(1).rolling(w).mean()
    return df


@dataclass
class XgbConfig:
    params: dict = field(default_factory=dict)
    min_observations: int = MIN_OBSERVATIONS


def fit_and_forecast(series: pd.Series, horizon: int, config: XgbConfig):
    """Return (predictions, info_dict). Falls back to persistence on short/failed series."""
    clean = series.sort_index().astype(float).asfreq("h")
    if clean.isna().all():
        return [float("nan")] * horizon, {}
    clean = clean.interpolate(limit_direction="both").ffill().bfill()
    if len(clean) < config.min_observations:
        last = float(clean.iloc[-1])
        return [last] * horizon, {}

    feat = make_features(clean).dropna()
    if len(feat) < 50:
        last = float(clean.iloc[-1])
        return [last] * horizon, {}

    X_train = feat[FEATURE_COLS]
    y_train = feat["y"]

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = XGBRegressor(**{**FIXED_PARAMS, **config.params})
            model.fit(X_train, y_train)

        # Recursive multi-step forecast: append each prediction so it can act as a lag.
        history = clean.copy()
        preds = []
        for _ in range(horizon):
            next_time = history.index[-1] + pd.Timedelta(hours=1)
            extended = history.copy()
            extended.loc[next_time] = np.nan
            frow = make_features(extended).iloc[[-1]][FEATURE_COLS]
            yhat = float(model.predict(frow)[0])
            preds.append(yhat)
            history.loc[next_time] = yhat

        info = {"train_rows": int(len(feat)), "n_features": len(FEATURE_COLS)}
        return preds, info
    except Exception as e:
        return [float("nan")] * horizon, {"error": str(e)}


print(f"Helper functions defined. {len(FEATURE_COLS)} features:")
print(FEATURE_COLS)

In [ ]:
# Metric helpers

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, float), np.array(y_pred, float)
    mask = y_true != 0
    if not mask.any():
        return float("nan")
    return float(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]).mean() * 100)


def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, float), np.array(y_pred, float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    if not mask.any():
        return float("nan")
    return float((np.abs(y_true[mask] - y_pred[mask]) / denom[mask]).mean() * 100)


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true, float) - np.array(y_pred, float)) ** 2)))


def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.array(y_true, float) - np.array(y_pred, float))))


print("Metric helpers defined.")

In [ ]:
# Grid search
import itertools
from tqdm.auto import tqdm

group_cols = ["region", "series"]

# Filter regions
df = load_df.copy()
if REGIONS is not None:
    df = df[df["region"].isin(REGIONS)].copy()

# Build all hyperparameter combos
param_grid = list(itertools.product(
    N_ESTIMATORS_VALUES,
    MAX_DEPTH_VALUES,
    LEARNING_RATE_VALUES,
    SUBSAMPLE_VALUES,
    COLSAMPLE_VALUES,
    MIN_CHILD_WEIGHT_VALUES,
))
n_series = df.groupby(group_cols).ngroups
print(f"{len(param_grid)} hyperparameter combinations x {n_series} series")
print(f"Total fits: {len(param_grid) * n_series}\n")

results = []

for n_est, depth, lr, subsample, colsample, min_child in tqdm(param_grid, desc="Param combos"):
    params = {
        "n_estimators": n_est,
        "max_depth": depth,
        "learning_rate": lr,
        "subsample": subsample,
        "colsample_bytree": colsample,
        "min_child_weight": min_child,
    }
    config = XgbConfig(params=params)
    label = f"n={n_est} d={depth} lr={lr} ss={subsample} cs={colsample} mcw={min_child}"

    series_results = []

    for group_keys, gdf in df.groupby(group_cols):
        region, series_name = group_keys

        ts = gdf.set_index("period")["value"]
        ts = ts[~ts.index.duplicated(keep="last")].sort_index().asfreq("h")
        non_null = ts.dropna()

        if len(non_null) <= HOLDOUT_HOURS:
            continue

        # Build train / test split
        test_start = non_null.index.max() - pd.Timedelta(hours=HOLDOUT_HOURS - 1)
        train_end = test_start - pd.Timedelta(hours=1)
        train_ts = ts.loc[:train_end]
        if MAX_TRAIN_HOURS is not None:
            cutoff = train_end - pd.Timedelta(hours=MAX_TRAIN_HOURS - 1)
            train_ts = train_ts.loc[cutoff:train_end]

        y_true = non_null.loc[test_start:].astype(float).values[:HOLDOUT_HOURS]
        preds, info = fit_and_forecast(train_ts, HOLDOUT_HOURS, config)
        y_pred = np.array(preds, float)

        if np.isnan(y_pred).any() or len(y_true) != len(y_pred):
            continue

        series_results.append({
            "region": region,
            "series": series_name,
            "mae": mae(y_true, y_pred),
            "rmse": rmse(y_true, y_pred),
            "mape": mape(y_true, y_pred),
            "smape": smape(y_true, y_pred),
            "train_rows": info.get("train_rows"),
        })

    if not series_results:
        continue

    sdf = pd.DataFrame(series_results)
    results.append({
        "params": label,
        "n_estimators": n_est,
        "max_depth": depth,
        "learning_rate": lr,
        "subsample": subsample,
        "colsample_bytree": colsample,
        "min_child_weight": min_child,
        "n_series": len(sdf),
        "mae": sdf["mae"].mean(),
        "rmse": sdf["rmse"].mean(),
        "mape": sdf["mape"].mean(),
        "smape": sdf["smape"].mean(),
    })

results_df = pd.DataFrame(results).sort_values("mape").reset_index(drop=True)
print(f"\nDone. {len(results_df)} hyperparameter combinations evaluated successfully.")

In [ ]:
# Results table - sorted by MAPE (lower is better)
display_cols = ["params", "n_series", "mae", "rmse", "mape", "smape"]

styled = (
    results_df[display_cols]
    .style
    .format({"mae": "{:.1f}", "rmse": "{:.1f}", "mape": "{:.3f}%", "smape": "{:.3f}%"})
    .background_gradient(subset=["mape"], cmap="RdYlGn_r")
    .set_caption(f"Grid search results - {HOLDOUT_HOURS}h holdout, sorted by MAPE ascending")
)
display(styled)

In [ ]:
# Best parameters summary
best = results_df.iloc[0]
print("=" * 60)
print("BEST PARAMETERS")
print("=" * 60)
print(f"  n_estimators      : {int(best.n_estimators)}")
print(f"  max_depth         : {int(best.max_depth)}")
print(f"  learning_rate     : {best.learning_rate}")
print(f"  subsample         : {best.subsample}")
print(f"  colsample_bytree  : {best.colsample_bytree}")
print(f"  min_child_weight  : {int(best.min_child_weight)}")
print(f"  Avg MAPE across all series : {best.mape:.3f}%")
print(f"  Avg MAE  across all series : {best.mae:.1f}")
print(f"  Avg RMSE across all series : {best.rmse:.1f}")
print()
print("Suggested production config (e.g. in your .env):")
print(f"  XGB_N_ESTIMATORS={int(best.n_estimators)}")
print(f"  XGB_MAX_DEPTH={int(best.max_depth)}")
print(f"  XGB_LEARNING_RATE={best.learning_rate}")
print(f"  XGB_SUBSAMPLE={best.subsample}")
print(f"  XGB_COLSAMPLE_BYTREE={best.colsample_bytree}")
print(f"  XGB_MIN_CHILD_WEIGHT={int(best.min_child_weight)}")

In [ ]:
# Bar chart - top 15 configs by MAPE
import matplotlib.pyplot as plt

top15 = results_df.head(15).copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

colors = ["#2ecc71" if i == 0 else "#95a5a6" for i in range(len(top15))]

axes[0].barh(top15["params"][::-1], top15["mape"][::-1], color=colors[::-1])
axes[0].set_xlabel("MAPE (%) - lower is better")
axes[0].set_title("Top 15 configurations by MAPE")
axes[0].axvline(top15["mape"].iloc[0], color="green", linestyle="--", alpha=0.6)

axes[1].barh(top15["params"][::-1], top15["rmse"][::-1], color=colors[::-1])
axes[1].set_xlabel("RMSE - lower is better")
axes[1].set_title("Top 15 configurations by RMSE")

fig.suptitle(f"XGBoost Grid Search - {HOLDOUT_HOURS}h holdout", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("xgboost_grid_search.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: xgboost_grid_search.png")

In [ ]:
# Forecast vs actual plots for the best config
# Shows up to 4 regions side by side

best_params = {
    "n_estimators": int(best.n_estimators),
    "max_depth": int(best.max_depth),
    "learning_rate": float(best.learning_rate),
    "subsample": float(best.subsample),
    "colsample_bytree": float(best.colsample_bytree),
    "min_child_weight": int(best.min_child_weight),
}
best_config = XgbConfig(params=best_params)

plot_regions = sorted(df["region"].unique())[:4]
fig, axes = plt.subplots(len(plot_regions), 1, figsize=(14, 4 * len(plot_regions)), sharex=False)
if len(plot_regions) == 1:
    axes = [axes]

for ax, region in zip(axes, plot_regions):
    region_df = df[df["region"] == region]
    # Use the first series found for this region
    series_name = region_df["series"].iloc[0]
    gdf = region_df[region_df["series"] == series_name]

    ts = gdf.set_index("period")["value"]
    ts = ts[~ts.index.duplicated(keep="last")].sort_index().asfreq("h")
    non_null = ts.dropna()

    if len(non_null) <= HOLDOUT_HOURS:
        ax.set_title(f"{region} - insufficient data")
        continue

    test_start = non_null.index.max() - pd.Timedelta(hours=HOLDOUT_HOURS - 1)
    train_end = test_start - pd.Timedelta(hours=1)
    train_ts = ts.loc[:train_end]
    if MAX_TRAIN_HOURS is not None:
        cutoff = train_end - pd.Timedelta(hours=MAX_TRAIN_HOURS - 1)
        train_ts = train_ts.loc[cutoff:]

    y_true = non_null.loc[test_start:].values[:HOLDOUT_HOURS]
    preds, _ = fit_and_forecast(train_ts, HOLDOUT_HOURS, best_config)
    y_pred = np.array(preds, float)

    # Show 7 days of history + the holdout window
    history_start = train_end - pd.Timedelta(days=7)
    history_ts = ts.loc[history_start:train_end]
    holdout_idx = pd.date_range(start=test_start, periods=HOLDOUT_HOURS, freq="h")

    ax.plot(history_ts.index, history_ts.values, color="steelblue", label="actual (history)")
    ax.plot(holdout_idx, y_true, color="steelblue", linestyle="--", label="actual (holdout)")
    ax.plot(holdout_idx, y_pred, color="tomato", linewidth=2, label="forecast (XGBoost)")
    ax.axvline(test_start, color="gray", linestyle=":", alpha=0.8)
    ax.set_title(f"{region} - {series_name}  |  MAPE={mape(y_true, y_pred):.2f}%")
    ax.set_ylabel("MW")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Time (UTC)")
fig.suptitle(f"Best config: {best.params} - {HOLDOUT_HOURS}h holdout", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("xgboost_best_forecast.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: xgboost_best_forecast.png")

In [ ]:
# Save the full results table as a CSV for reference
results_df.to_csv("xgboost_grid_search_results.csv", index=False)
print("Results saved to xgboost_grid_search_results.csv")

# Download all outputs
from google.colab import files
files.download("xgboost_grid_search_results.csv")
files.download("xgboost_grid_search.png")
files.download("xgboost_best_forecast.png")